# C2 · Train and explain our first model

**Course:** Business Intelligence · IIB423T-1 · Universidad del Desarrollo
**Group:** 01 · Alessandro Lavezzi · José Saavedra
**Data source:** monthly payroll files of fixed-term (*contrata*) staff published by six Santiago Metropolitan Health Services on Chile's Transparency Portal (Ley 20.285, Transparencia Activa), September 2024 – August 2026.

**Route:** classification.

**Question.** On 1 September 2025, using only what was known up to that day, can we anticipate which people will show persistent overtime in the following 12 months?

**How to run.** This notebook first runs `W2_project.ipynb` (same folder) to reuse its data preparation and its person-level train/validation/test split. Then it builds the C2 dataset, a simple reference rule and a logistic regression model. Every random step uses `SEED = 42`.

In [1]:
%%capture w2_log
# 0.1 REUSE THE W2 PREPARATION AND SPLIT
%run W2_project.ipynb

In [2]:
# 0.2 CHECK THAT THE W2 OBJECTS ARE AVAILABLE
print("W2 objects loaded:", all(n in globals() for n in ["nomina_v", "pm_all", "particion", "SEED"]))
print(f"Person-months: {len(pm_all):,} | people in the W2 split: {len(particion):,}")
print(particion["conjunto"].value_counts().to_string())

W2 objects loaded: True
Person-months: 230,053 | people in the W2 split: 16,791
conjunto
entrenamiento    11757
validacion        2519
prueba            2515


## 1. C2 dataset

### 1.1 Year 1 features

The prediction is made on **1 September 2025**. Every feature is computed only from months up to **August 2025** (year 1), so the model never sees information from the period it predicts.

| Feature | Definition | Why |
|---|---|---|
| `meses_x` | months on the payroll in year 1 | how much history we have for the person |
| `pct_meses_hhee_x` | % of year-1 months with overtime | past persistence: the strongest expected signal |
| `horas_prom_x` | mean overtime hours per month in year 1 (0 in months without overtime) | intensity of overtime use |
| `estamento_x` | staff group in the **last month** of year 1 | the W2 leakage audit showed that the most frequent staff group over the whole window uses later months |
| `derecho_x` | 1 if entitled to overtime in the last month of year 1 | administrative condition known at prediction time |
| `servicio` | health service | differences between services found in W2 |

The share of overtime in gross pay is not used: it has missing values when gross pay is zero and impossible values (flagged in W2), and `horas_prom_x` already measures intensity.

In [3]:
# 1.1 YEAR 1 FEATURES: ONLY WHAT WAS KNOWN ON 1 SEPTEMBER 2025
CLAVE_P = ["servicio", "Nombre completo"]          # one person = service + name (same rule as W2)
FIN_ANIO_1 = "2025-08"                             # last month available at prediction time

# staff group and overtime entitlement of each person-month (first record if a person has several contracts)
info_mes = (nomina_v.groupby(CLAVE)
                    .agg(estamento_mes=("Estamento", "first"),
                         derecho_mes=("Derecho a horas extraordinarias", "first"))
                    .reset_index())
info_mes["estamento_mes"] = info_mes["estamento_mes"].replace(NOMBRE_CORTO)

pm_c2 = pm_all[CLAVE + ["horas_hhee", "tiene_hhee"]].merge(info_mes, on=CLAVE, how="left")
anio_1 = pm_c2[pm_c2["periodo"] <= FIN_ANIO_1]

# numeric features: summaries of year 1 per person
X_anio_1 = anio_1.groupby(CLAVE_P).agg(
    meses_x=("periodo", "nunique"),
    pct_meses_hhee_x=("tiene_hhee", "mean"),
    horas_prom_x=("horas_hhee", "mean"),
)
X_anio_1["pct_meses_hhee_x"] *= 100

# categorical features: value in the LAST month of year 1 (not the most frequent value over the whole window)
ultimo_mes = anio_1.sort_values("periodo").groupby(CLAVE_P).tail(1).set_index(CLAVE_P)
X_anio_1["estamento_x"] = ultimo_mes["estamento_mes"]
X_anio_1["derecho_x"] = ultimo_mes["derecho_mes"].eq("Sí").astype(int)
X_anio_1 = X_anio_1.reset_index()

print(f"People with at least one month in year 1: {len(X_anio_1):,}")
print("Last month used:", anio_1["periodo"].max(), "| missing values per column:")
print(X_anio_1.isna().sum().to_string())
display(X_anio_1.describe().round(1))

People with at least one month in year 1: 12,065
Last month used: 2025-08 | missing values per column:
servicio            0
Nombre completo     0
meses_x             0
pct_meses_hhee_x    0
horas_prom_x        0
estamento_x         0
derecho_x           0


,meses_x,pct_meses_hhee_x,horas_prom_x,derecho_x
count,12065.000,12065.000,12065.000,12065.000
mean,9.400,23.400,10.100,0.300
std,3.800,35.500,24.000,0.400
min,1.000,0.000,0.000,0.000
25%,7.000,0.000,0.000,0.000
50%,12.000,0.000,0.000,0.000
75%,12.000,44.400,7.500,1.000
max,12.000,100.000,267.500,1.000


### 1.2 Year 2 target and eligible people

**Target (`persistente_y`).** A person is *persistent* in year 2 (September 2025 – August 2026) if they received overtime in at least 75% of their year-2 months. This is the same threshold as the project KPI since C1. **Positive class = persistent (1).**

**Eligible people.** We keep people who appear in **both** years with **at least 6 months in each**. With fewer months, a share of months with overtime is too unstable (for example, 1 of 1 month = 100%). Six months is half a year: enough history to describe the person, without excluding people who joined or left during the year.

One row = one person (service + name).

In [4]:
# 1.2 YEAR 2 TARGET AND ELIGIBLE PEOPLE
INICIO_ANIO_2 = "2025-09"                          # first month of the period we predict
MESES_MIN_C2 = 6                                   # minimum months on the payroll in EACH year

# target: persistent in year 2 = overtime in at least UMBRAL_PERSISTENCIA % of the person's year-2 months
anio_2 = pm_c2[pm_c2["periodo"] >= INICIO_ANIO_2]
Y_anio_2 = anio_2.groupby(CLAVE_P).agg(
    meses_y=("periodo", "nunique"),
    pct_meses_hhee_y=("tiene_hhee", "mean"),
).reset_index()
Y_anio_2["pct_meses_hhee_y"] *= 100
Y_anio_2["persistente_y"] = (Y_anio_2["pct_meses_hhee_y"] >= UMBRAL_PERSISTENCIA).astype(int)

# keep people observed in both years, with enough months in each
datos_c2 = X_anio_1.merge(Y_anio_2, on=CLAVE_P, how="inner")
elegible = (datos_c2["meses_x"] >= MESES_MIN_C2) & (datos_c2["meses_y"] >= MESES_MIN_C2)

print("Year 2 months:", anio_2["periodo"].min(), "to", anio_2["periodo"].max())
print(f"People in year 1: {len(X_anio_1):,} | in year 2: {len(Y_anio_2):,} | in both: {len(datos_c2):,} "
      f"| with ≥{MESES_MIN_C2} months in each year: {elegible.sum():,}")
datos_c2 = datos_c2[elegible].reset_index(drop=True)
print(f"Persistent in year 2 (positive class): {datos_c2['persistente_y'].sum():,} "
      f"of {len(datos_c2):,} ({datos_c2['persistente_y'].mean():.1%})")

Year 2 months: 2025-09 to 2026-08
People in year 1: 12,065 | in year 2: 14,635 | in both: 9,909 | with ≥6 months in each year: 8,279
Persistent in year 2 (positive class): 1,524 of 8,279 (18.4%)


### 1.3 Reuse of the W2 split

We do **not** create a new split. Each person keeps the set assigned in W2 (section 6.1): split by person (service + name), stratified by service and staff group, 70/15/15, `SEED = 42`. This way the same person is never in training and in test, and the test set stays the one fixed in W2.

| Set | Use in C2 |
|---|---|
| Training (`entrenamiento`) | fit scaling, encoding and the model coefficients |
| Validation (`validacion`) | compare the model with the reference and choose the decision threshold |
| Test (`prueba`) | final evaluation, **used once**, after the model and threshold are fixed |

The C2 dataset is a subset of the W2 people (only those with ≥ 6 months in each year), so the shares are close to, but not exactly, 70/15/15.

In [5]:
# 1.3 REUSE THE W2 SPLIT: EACH PERSON KEEPS THE SET ASSIGNED IN W2
datos_c2 = datos_c2.merge(particion[CLAVE_P + ["conjunto"]], on=CLAVE_P, how="left")
print("People without a set:", datos_c2["conjunto"].isna().sum(),
      "| people repeated:", datos_c2.duplicated(CLAVE_P).sum())

ORDEN_CONJ = ["entrenamiento", "validacion", "prueba"]     # training, validation, test
resumen_c2 = datos_c2.groupby("conjunto").agg(
    people=("persistente_y", "size"),
    persistent=("persistente_y", "sum"),
    prevalence_pct=("persistente_y", "mean"),
).reindex(ORDEN_CONJ)
resumen_c2["prevalence_pct"] *= 100
resumen_c2["share_of_people_pct"] = resumen_c2["people"] / resumen_c2["people"].sum() * 100
display(resumen_c2.round(1))

People without a set: 0 | people repeated: 0


,people,persistent,prevalence_pct,share_of_people_pct
conjunto,,,,
entrenamiento,5813,1071,18.400,70.200
validacion,1255,243,19.400,15.200
prueba,1211,210,17.300,14.600


## 2. Evaluation tools

### 2.1 Metrics

**Positive class = persistent (1).** For each set of predictions we compute the confusion matrix and:

| Metric | Formula | Question it answers in this project |
|---|---|---|
| Precision | TP / (TP + FP) | Of the people we flag as persistent, how many really are? |
| Recall | TP / (TP + FN) | Of the people who really are persistent, how many do we find? |
| F1 | 2 · precision · recall / (precision + recall) | Balance between both, in one number |
| Accuracy | (TP + TN) / total | Share of correct predictions, both classes together |

**Why accuracy alone is not enough.** Only about 18% of people are persistent. A rule that predicts "nobody is persistent" is right for every non-persistent person and therefore reaches about 82% accuracy, while finding none of the people we care about (recall = 0). Precision, recall and F1 focus on the positive class.

The same functions are used for the model and for the reference, so both are measured in exactly the same way. They are first checked on a small example whose result is known by hand.

In [6]:
# 2.1 EVALUATION FUNCTIONS: THE SAME CODE MEASURES THE MODEL AND THE REFERENCE
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def metricas(y_real, y_pred, nombre):
    """Confusion-matrix counts and metrics for the positive class (1 = persistent)."""
    vn, fp, fn, vp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return pd.Series({
        "TP": vp, "FP": fp, "FN": fn, "TN": vn,
        "precision": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "F1": f1_score(y_real, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_real, y_pred),
    }, name=nombre)

def tabla_confusion(y_real, y_pred):
    """Confusion matrix as a labelled table: rows = actual, columns = predicted."""
    m = confusion_matrix(y_real, y_pred, labels=[1, 0])
    return pd.DataFrame(m, index=["Actual: persistent", "Actual: not persistent"],
                        columns=["Predicted: persistent", "Predicted: not persistent"])

# check 1: toy example whose result we know by hand (TP=2, FN=1, FP=1, TN=6)
y_toy_real = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0])
y_toy_pred = np.array([1, 1, 0, 1, 0, 0, 0, 0, 0, 0])
print("Toy example (expected: precision = recall = F1 = 0.667, accuracy = 0.800):")
display(tabla_confusion(y_toy_real, y_toy_pred))
display(metricas(y_toy_real, y_toy_pred, "toy example").to_frame().T.round(3))

# check 2: why accuracy alone is not enough (training set only; test is not used here)
y_ent = datos_c2.loc[datos_c2["conjunto"] == "entrenamiento", "persistente_y"]
print("Rule 'nobody is persistent' on the training set:")
display(metricas(y_ent, np.zeros(len(y_ent), dtype=int), "always 'not persistent'").to_frame().T.round(3))

Toy example (expected: precision = recall = F1 = 0.667, accuracy = 0.800):


,Predicted: persistent,Predicted: not persistent
Actual: persistent,2,1
Actual: not persistent,1,6


,TP,FP,FN,TN,precision,recall,F1,accuracy
toy example,2.000,1.000,1.000,6.000,0.667,0.667,0.667,0.800


Rule 'nobody is persistent' on the training set:


,TP,FP,FN,TN,precision,recall,F1,accuracy
always 'not persistent',0.000,0.000,1071.000,4742.000,0.000,0.000,0.000,0.816


### 2.2 Reference rule (baseline)

**Rule.** Predict *persistent* in year 2 if the person had overtime in at least 75% of their year-1 months; otherwise predict *not persistent*. In words: "whoever was persistent last year will be persistent this year".

**Why this reference.** It is the simplest prediction a manager could make with the same information, without any model. It uses one variable (`pct_meses_hhee_x`) and the same 75% threshold as the project KPI, so it has nothing to fit: no training data are used to build it.

It is measured on the **validation** set here, with the same functions that will measure the model. The test set is not used until the model and its threshold are fixed.

In [7]:
# 2.2 REFERENCE RULE: "PERSISTENT IN YEAR 1 -> PERSISTENT IN YEAR 2"
def regla_referencia(datos):
    """Predict persistent (1) if the person had overtime in at least UMBRAL_PERSISTENCIA % of year-1 months."""
    return (datos["pct_meses_hhee_x"] >= UMBRAL_PERSISTENCIA).astype(int)

ent_c2 = datos_c2[datos_c2["conjunto"] == "entrenamiento"]
val_c2 = datos_c2[datos_c2["conjunto"] == "validacion"]

y_val = val_c2["persistente_y"]
pred_ref_val = regla_referencia(val_c2)

print(f"Reference rule on the validation set ({len(val_c2):,} people, {y_val.sum()} persistent):")
display(tabla_confusion(y_val, pred_ref_val))
resultados_val = pd.DataFrame([metricas(y_val, pred_ref_val, "reference")])
display(resultados_val.round(3))

Reference rule on the validation set (1,255 people, 243 persistent):


,Predicted: persistent,Predicted: not persistent
Actual: persistent,181,62
Actual: not persistent,67,945


,TP,FP,FN,TN,precision,recall,F1,accuracy
reference,181.000,67.000,62.000,945.000,0.730,0.745,0.737,0.897


## 3. Model

### 3.1 Logistic regression

**Why this model.** Logistic regression is simple, fast and explainable: it combines the features with one weight (coefficient) each and turns the result into a probability between 0 and 1 of being persistent. Its coefficients can be read and discussed, which matters more here than a small gain in performance. The brief states that a simple model is enough.

**Main settings.**

| Setting | Value | Why |
|---|---|---|
| Numeric features | `meses_x`, `pct_meses_hhee_x`, `horas_prom_x`, `derecho_x` | year-1 information only |
| Categorical features | `estamento_x`, `servicio` | one-hot encoded, first category dropped as reference |
| Scaling | `StandardScaler` (mean 0, standard deviation 1) | puts features on the same scale so that no feature dominates because of its units |
| Penalty | L2, `C = 1` (scikit-learn default) | mild regularization that keeps coefficients stable; no tuning |
| Decision threshold | chosen in validation (section 3.2) | the default 0.5 is not assumed to be the best for an 18% positive class |

**Fair preparation.** Scaling, encoding and coefficients are fitted inside one `Pipeline`, with `fit` called **only on the training set** (5,813 people). Validation and test people are only transformed with what was learned on training.

In [8]:
# 3.1 LOGISTIC REGRESSION: PREPARATION AND COEFFICIENTS FITTED ON TRAINING ONLY
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

VARS_NUM = ["meses_x", "pct_meses_hhee_x", "horas_prom_x", "derecho_x"]   # numeric year-1 features
VARS_CAT = ["estamento_x", "servicio"]                                     # categorical features
OBJETIVO = "persistente_y"                                                 # target (1 = persistent)

preparacion = ColumnTransformer([
    ("num", StandardScaler(), VARS_NUM),                                       # mean 0, std 1
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), VARS_CAT),  # one 0/1 column per category
])
modelo = Pipeline([
    ("prep", preparacion),
    ("logit", LogisticRegression(max_iter=1000, random_state=SEED)),           # default L2 penalty, C = 1
])

X_ent = ent_c2[VARS_NUM + VARS_CAT]
y_ent = ent_c2[OBJETIVO]
modelo.fit(X_ent, y_ent)                                                       # the ONLY fit in the notebook

# evidence: the scaler learned its means from the training rows only
escalador = modelo.named_steps["prep"].named_transformers_["num"]
display(pd.DataFrame({
    "mean_learned_by_scaler": escalador.mean_,
    "mean_training": X_ent[VARS_NUM].mean().values,
    "mean_all_C2_people": datos_c2[VARS_NUM].mean().values,
}, index=VARS_NUM).round(3))

codificador = modelo.named_steps["prep"].named_transformers_["cat"]
for var, cats in zip(VARS_CAT, codificador.categories_):
    print(f"{var}: reference category = {cats[0]} | other categories = {list(cats[1:])}")
print(f"Model fitted on {len(X_ent):,} training people ({y_ent.sum():,} persistent) "
      f"with {len(modelo.named_steps['logit'].coef_[0])} coefficients + intercept.")

,mean_learned_by_scaler,mean_training,mean_all_C2_people
meses_x,11.434,11.434,11.427
pct_meses_hhee_x,26.514,26.514,26.456
horas_prom_x,10.380,10.380,10.438
derecho_x,0.295,0.295,0.296


estamento_x: reference category = Administrativo | other categories = ['Auxiliar', 'Directivo', 'Médicos y afines', 'Profesional', 'Técnico']
servicio: reference category = Centro | other categories = ['Norte', 'Occidente', 'Oriente', 'Sur', 'SurOriente']
Model fitted on 5,813 training people (1,071 persistent) with 14 coefficients + intercept.
